In [12]:
import os
import re
import json
import glob
import pandas as pd
import math

# ==========================================================
# Configuration
# ==========================================================

OUTPUT_GEOJSON = "DataEngineeredClimateNormals.geojson"
ZIP_CENTROID_CSV = "WI_PWC.csv"

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

VARIABLES = [
    "avg_temp_f",
    "avg_max_temp_f",
    "avg_min_temp_f",
    "precip_in",
    "snowfall_in"
]

NUMERIC_COLUMNS = [
    "Average temperature (F)",
    "Average maximum temperature (F)",
    "Average minimum temperature (F)",
    "Precipitation (in)",
    "Snowfall (in)"
]

# ==========================================================
# Load WI population-weighted centroids
# ==========================================================

pw_df = pd.read_csv(ZIP_CENTROID_CSV, dtype={'STD_ZIP5': str})
zip_to_coords = {
    row['STD_ZIP5']: [row['LONGITUDE'], row['LATITUDE']]
    for _, row in pw_df.iterrows()
}

# ==========================================================
# Initialize Fresh GeoJSON
# ==========================================================

geojson = {
    "type": "FeatureCollection",
    "metadata": {
        "months": MONTHS,
        "variables": VARIABLES
    },
    "features": []
}

existing_zips = set()

# ==========================================================
# Helper: Extract ZIPs from filename
# ==========================================================

def extract_zips_from_filename(filename):
    return re.findall(r'_(\d{5})', f"_{filename}_")

# ==========================================================
# Process CSV Files
# ==========================================================

csv_files = glob.glob("*.csv")

for csv_file in csv_files:

    zips = extract_zips_from_filename(csv_file)
    if not zips:
        continue

    print(f"Processing {csv_file}: ZIPs {zips}")

    df = pd.read_csv(csv_file)

    for col in NUMERIC_COLUMNS:
        if col not in df.columns:
            df[col] = None

    df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")

    if len(df) != 12:
        print(f"Warning: {csv_file} does not contain 12 rows. Skipping.")
        continue

    # Build climate array (12x5) with NaN -> None
    climate_array = []
    for _, row in df.iterrows():
        row_values = []
        for col in NUMERIC_COLUMNS:
            val = row[col]
            if val is None or (isinstance(val, float) and math.isnan(val)):
                row_values.append(None)
            else:
                row_values.append(float(val))
        climate_array.append(row_values)

    # Create feature per ZIP
    for zip_code in zips:
        if zip_code in existing_zips:
            print(f"Duplicate ZIP detected: {zip_code}, skipping")
            continue

        coords = zip_to_coords.get(zip_code, [0,0])  # fallback

        feature = {
            "type": "Feature",
            "properties": {"zip": zip_code, "climate": climate_array},
            "geometry": {"type": "Point", "coordinates": coords}
        }

        geojson["features"].append(feature)
        existing_zips.add(zip_code)

# ==========================================================
# Write GeoJSON
# ==========================================================

with open(OUTPUT_GEOJSON, "w") as f:
    json.dump(geojson, f, indent=2, allow_nan=False)

print(f"Fresh GeoJSON written to {OUTPUT_GEOJSON}, {len(existing_zips)} ZIPs total")

Processing Belgium_53004_53013.csv: ZIPs ['53004', '53013']
Processing Brillion_54207.csv: ZIPs ['54207']
Processing Chilton_53042.csv: ZIPs ['53042']
Processing Germantown_53097.csv: ZIPs ['53097']
Processing Grafton_53012_53024_53092.csv: ZIPs ['53012', '53024', '53092']
Processing Hingham_53031_53093.csv: ZIPs ['53031', '53093']
Processing Manitowac_53063.csv: ZIPs ['53063']
Processing Oostburg_53070.csv: ZIPs ['53070']
Processing Plymouth5_3_53011_53023_53703.csv: ZIPs ['53011', '53023', '53703']
Processing PlymouthWWTP_53020_53026_53073.csv: ZIPs ['53020', '53026', '53073']
Processing PortWashington_53074.csv: ZIPs ['53074']
Processing RandomLake_53021_53001_53075.csv: ZIPs ['53021', '53001', '53075']
Processing Saukville_53080.csv: ZIPs ['53080']
Processing Sheboygan3_2_53083_53015.csv: ZIPs ['53083', '53015']
Processing SheboyganMemorial_53085.csv: ZIPs ['53085']
Processing Sheboygan_53044_53081_53082.csv: ZIPs ['53044', '53081', '53082']
Processing TwoRivers4_5_54214.csv: ZIPs 

In [10]:
import pandas as pd

# Load full population-weighted centroids
pw_df = pd.read_csv("ZIP_Code_Population_Weighted_Centroids.csv", dtype={'STD_ZIP5': str})

# Keep only Wisconsin ZIPs (state = 'WI')
wi_pw_df = pw_df[pw_df['USPS_ZIP_PREF_STATE_1221'] == 'WI']

# Keep only the columns we need for mapping
wi_pw_df = wi_pw_df[['STD_ZIP5', 'LATITUDE', 'LONGITUDE']]

# Save as WI_PWC.csv
wi_pw_df.to_csv("WI_PWC.csv", index=False)

print(f"Saved {len(wi_pw_df)} Wisconsin ZIP codes to WI_PWC.csv")

Saved 721 Wisconsin ZIP codes to WI_PWC.csv
